## Libraries

In [1]:
import xarray as xr
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

from pyproj import Transformer

## Data Cleaning

First the needed columns of the fire dataframe are extracted and converted to their corresponding datatype.

In [2]:
df = pd.read_excel("data/fire_data.xlsx")

starting_rows = len(df)

numeric_cols = ['Huso', 'CoordenadaX', 'CoordenadaY']

other_cols = ['Detectado', 'Datum']

for c in numeric_cols:
    df[c] = df[c].astype(str).str.replace(',', '.')
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['Detectado'] = pd.to_datetime(df['Detectado'], errors='coerce', format='%d/%m/%Y %H:%M:%S').dt.normalize()


df.dropna(subset=['CoordenadaX', 'CoordenadaY'], inplace=True)
df.drop(columns = [c for c in df.columns if c not in numeric_cols + other_cols], inplace=True)

df = df[df["Huso"].between(29.0, 31.0)]


The coordinates are in two different formats (EPSG:23030 for the old data and EPSG:25830 for the new data). Both formats are converted into EPSG:4326. Some rows have manual errors, a wide range is aplied to clean outliners values.

In [3]:
old_29_transformer = Transformer.from_crs("EPSG:23029", "EPSG:4326", always_xy=True)
old_30_transformer = Transformer.from_crs("EPSG:23030", "EPSG:4326", always_xy=True)
old_31_transformer = Transformer.from_crs("EPSG:23031", "EPSG:4326", always_xy=True)
new_29_transformer = Transformer.from_crs("EPSG:25829", "EPSG:4326", always_xy=True)
new_30_transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
new_31_transformer = Transformer.from_crs("EPSG:25831", "EPSG:4326", always_xy=True)

transformers = {
    (29, False): old_29_transformer,
    (30, False): old_30_transformer,
    (31, False): old_31_transformer,
    (29, True):  new_29_transformer,
    (30, True):  new_30_transformer,
    (31, True):  new_31_transformer,
}

mask_etrs = df['Datum'].astype(str).str.contains('ETRS89', case=False, na=False)

df['Longitude'] = float('nan')
df['Latitude'] = float('nan')

for (huso, is_etrs), group in df.groupby(['Huso', mask_etrs]):
    transformer = transformers[(int(huso), bool(is_etrs))]
    lon, lat = transformer.transform(
        group['CoordenadaX'].values,
        group['CoordenadaY'].values
    )
    df.loc[group.index, 'Longitude'] = lon
    df.loc[group.index, 'Latitude'] = lat


df = df[df['Longitude'].between(-8, -1.5) & df['Latitude'].between(39.5, 43.5)]

df.drop(columns=['Huso', 'CoordenadaX', 'CoordenadaY', 'Datum'], inplace=True)

print(f"Removed rows: {starting_rows - len(df)}")
print(f"Remaining rows: {len(df)}")

Removed rows: 1825
Remaining rows: 27537


Definition Of Borders And Time Limits

In [4]:
first_fire = df['Detectado'].min()
last_fire = df['Detectado'].max()

min_lat, max_lat = df['Latitude'].min(), df['Latitude'].max()
min_lon, max_lon = df['Longitude'].min(), df['Longitude'].max()

print(f"First fire: {first_fire}")
print(f"Last fire: {last_fire}")
print(f"Latitude range: {min_lat} to {max_lat}")
print(f"Longitude range: {min_lon} to {max_lon}")

start_time = first_fire - pd.Timedelta(days=90)
end_time = last_fire

First fire: 2006-01-21 00:00:00
Last fire: 2023-12-30 00:00:00
Latitude range: 40.090201004204765 to 43.23304854759588
Longitude range: -7.065729152572756 to -1.8077660689453114


Loading, Cleaning And Bounding Of The Climate Data

In [5]:
elevation_ds = xr.open_dataset("data/elev_ens_0.1deg_reg_v33.0e.nc")
wind_speed_ds = xr.open_dataset("data/fg_ens_mean_0.1deg_reg_v33.0e.nc")
rel_humidity_ds = xr.open_dataset("data/hu_ens_mean_0.1deg_reg_v33.0e.nc")
preasure_sl_ds = xr.open_dataset("data/pp_ens_mean_0.1deg_reg_v33.0e.nc")
radiation_ds = xr.open_dataset("data/qq_ens_mean_0.1deg_reg_v33.0e.nc")
rain_ds = xr.open_dataset("data/rr_ens_mean_0.1deg_reg_v33.0e.nc")
avg_temperature_ds = xr.open_dataset("data/tg_ens_mean_0.1deg_reg_v33.0e.nc")
min_temperature_ds = xr.open_dataset("data/tn_ens_mean_0.1deg_reg_v33.0e.nc")
max_temperature_ds = xr.open_dataset("data/tx_ens_mean_0.1deg_reg_v33.0e.nc")


elevation_ds = elevation_ds.sel(longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
wind_speed_ds = wind_speed_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
rel_humidity_ds = rel_humidity_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
preasure_sl_ds = preasure_sl_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
radiation_ds = radiation_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
rain_ds = rain_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
avg_temperature_ds = avg_temperature_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
min_temperature_ds = min_temperature_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
max_temperature_ds = max_temperature_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))

Loading Of The Vegetation Data

In [6]:
mfe = gpd.read_file("data/mfe_cyl/MFE_41.shp")

numeric_vars_mfe = [
    "Superficie", "FCCARB", "FCCMAT",
    "FCCHER", "FCCTOT", "HMMAT",
    "O1", "O2", "O3"
]

categorical_vars_mfe = [
    "TipesTR25", "Especie1", "Especie2",
    "Especie3", "Estado1", "Estado2",
    "Estado3", "Distribuci", "TIPOBOSQUE",
    "n_MODCOM", "n_REGBIO"
]


mfe = mfe[numeric_vars_mfe + categorical_vars_mfe + ["geometry"]]

Mapping Of Cat. Variables And Cleaning Of The File

In [7]:
mapa_riesgo_especies = {
    "sin datos": 0,
    
    # Nivel 1: Muy bajo (Ribera y medios húmedos)
    "Salix spp.": 1, "Salix atrocinerea": 1, "Salix alba": 1, "Salix fragilis": 1,
    "Salix caprea": 1, "Salix elaeagnos": 1, "Salix purpurea": 1, "Salix babylonica": 1,
    "Alnus glutinosa": 1, "Populus x canadensis": 1, "Populus nigra": 1, "Populus alba": 1,
    "Populus tremula": 1, "Fraxinus angustifolia": 1, "Fraxinus excelsior": 1, "Fraxinus ornus": 1,
    "Ulmus minor": 1, "Ulmus glabra": 1, "Ulmus pumila": 1, "Tamarix spp.": 1,
    "Sambucus nigra": 1, "Platanus hispanica": 1,
    
    # Nivel 2: Bajo (Frondosas caducifolias)
    "Fagus sylvatica": 2, "Ficus carica": 2, "Quercus pyrenaica": 2, "Quercus faginea": 2,
    "Quercus petraea": 2, "Quercus humilis (Q. Pubescens)": 2, "Quercus canariensis": 2,
    "Quercus robur": 2, "Quercus rubra": 2, "Castanea sativa": 2, "Betula alba": 2,
    "Betula spp.": 2, "Betula pendula": 2, "Acer campestre": 2, "Acer monspessulanum": 2,
    "Acer pseudoplatanus": 2, "Acer negundo": 2, "Acer opalus": 2, "Acer spp.": 2,
    "Sorbus aucuparia": 2, "Sorbus aria": 2, "Sorbus spp.": 2, "Sorbus torminalis": 2,
    "Prunus spp.": 2, "Prunus avium": 2, "Prunus padus": 2, "Corylus avellana": 2,
    "Juglans regia": 2, "Juglans nigra": 2, "Crataegus monogyna": 2, "Crataegus spp.": 2,
    "Celtis australis": 2, "Tilia spp.": 2, "Tilia platyphyllos": 2, "Tilia cordata": 2,
    "Pyrus spp.": 2, "Malus sylvestris": 2, "Robinia pseudoacacia": 2, "Ailanthus altissima": 2,
    "Carpinus betulus": 2, "Gleditsia triacanthos": 2, "Mezcla de pequeñas frondosas": 2,
    "Mezcla de frondosas de gran porte": 2, "Otras frondosas": 2,
    
    # Nivel 3: Medio (Perennifolias, esclerófilas y matorral arbóreo)
    "Quercus ilex": 3, "Quercus suber": 3, "Prunus lusitanica": 3, "Ilex aquifolium": 3,
    "Juniperus thurifera": 3, "Juniperus communis": 3, "Juniperus oxycedrus": 3,
    "Juniperus phoenicea": 3, "Arbutus unedo": 3, "Phillyrea latifolia": 3,
    "Olea europaea": 3, "Taxus baccata": 3,
    
    # Nivel 4: Alto (Coníferas estándar)
    "Pinus pinea": 4, "Pinus nigra": 4, "Pinus sylvestris": 4, "Pinus uncinata": 4,
    "Otra Coniferas": 4, "Mezcla de coníferas": 4, "Cupressus arizonica": 4,
    "Cupressus sempervirens": 4, "Cupressus macrocarpa": 4, "Pseudotsuga menziesii": 4,
    "Picea abies": 4, "Abies alba": 4, "Abies pinsapo": 4, "Thuja spp": 4,
    "Cedrus atlantica": 4, "Cedrus deodara": 4, "Chamaecyparis lawsoniana": 4,
    "Larix decidua": 4, "Larix spp.": 4, "Larix x eurolepis": 4,
    
    # Nivel 5: Extremo (Especies pirófilas de máxima propagación/carga)
    "Eucalyptus camaldulensis": 5, "Eucalyptus globulus": 5, "Eucalyptus nitens": 5,
    "Pinus halepensis": 5, "Pinus pinaster": 5, "Pinus radiata": 5,
    "Pinus canariensis": 5, "Acacia spp.": 5,
}

mapa_estado_ordinal = {
    "Sin datos": 0,
    "Fustal": 1,
    "Repoblado": 2,
    "Latizal": 3,
    "Monte bravo": 4,
}

mapa_tipo_bosque = {
    "-": 0,
    "Frondosas": 1,
    "Masas mixtas": 2,
    "Coníferas": 3,
}

mapa_distribucion = {
    "No procede": 0,
    "Piés aislados": 1,
    "Discontinua en bosquetes": 2,
    "Discontinua en fajas": 3,
    "Discontinua en mosaico": 4,
    "Discontinua irregular": 5,
    "Uniforme": 6,
}

mfe["Riesgo_Especie1"] = mfe["Especie1"].map(mapa_riesgo_especies).fillna(30)
mfe["Riesgo_Especie2"] = mfe["Especie2"].map(mapa_riesgo_especies).fillna(30)
mfe["Riesgo_Especie3"] = mfe["Especie3"].map(mapa_riesgo_especies).fillna(30)
mfe["Estado1_num"] = mfe["Estado1"].map(mapa_estado_ordinal).fillna(0)
mfe["Estado2_num"] = mfe["Estado2"].map(mapa_estado_ordinal).fillna(0)
mfe["Estado3_num"] = mfe["Estado3"].map(mapa_estado_ordinal).fillna(0)
mfe["TIPOBOSQUE_num"] = mfe["TIPOBOSQUE"].map(mapa_tipo_bosque).fillna(0)
mfe["Distribuci_num"] = mfe["Distribuci"].map(mapa_distribucion).fillna(0)

mfe.drop(columns=["Especie1", "Especie2", "Especie3", "Estado1", "Estado2", "Estado3", "TIPOBOSQUE", "Distribuci"], inplace=True)

mfe = mfe.to_crs("EPSG:4326")

## Feature Extraction

In [8]:
# 1. Precipitación acumulada (Estrés hídrico a corto, medio y largo plazo)
rain_ds['rr_7d_sum'] = rain_ds['rr'].rolling(time=7, min_periods=1).sum()
rain_ds['rr_30d_sum'] = rain_ds['rr'].rolling(time=30, min_periods=1).sum()
rain_ds['rr_60d_sum'] = rain_ds['rr'].rolling(time=60, min_periods=1).sum()
rain_ds['rr_90d_sum'] = rain_ds['rr'].rolling(time=90, min_periods=1).sum()

# 2. Humedad relativa media (Desecación sostenida del ambiente)
rel_humidity_ds['hu_7d_mean'] = rel_humidity_ds['hu'].rolling(time=7, min_periods=1).mean()
rel_humidity_ds['hu_14d_mean'] = rel_humidity_ds['hu'].rolling(time=14, min_periods=1).mean()
rel_humidity_ds['hu_30d_mean'] = rel_humidity_ds['hu'].rolling(time=30, min_periods=1).mean()

# 3. Temperatura máxima (Olas de calor y pico de evapo-transpiración)
max_temperature_ds['tx_3d_max'] = max_temperature_ds['tx'].rolling(time=3, min_periods=1).max()
max_temperature_ds['tx_7d_mean'] = max_temperature_ds['tx'].rolling(time=7, min_periods=1).mean()
max_temperature_ds['tx_14d_mean'] = max_temperature_ds['tx'].rolling(time=14, min_periods=1).mean()
max_temperature_ds['tx_30d_mean'] = max_temperature_ds['tx'].rolling(time=30, min_periods=1).mean()

# 4. Temperatura media acumulada
avg_temperature_ds['tg_7d_mean'] = avg_temperature_ds['tg'].rolling(time=7, min_periods=1).mean()
avg_temperature_ds['tg_30d_mean'] = avg_temperature_ds['tg'].rolling(time=30, min_periods=1).mean()

# 5. Radiación solar acumulada (Exposición directa)
radiation_ds['qq_7d_sum'] = radiation_ds['qq'].rolling(time=7, min_periods=1).sum()
radiation_ds['qq_30d_sum'] = radiation_ds['qq'].rolling(time=30, min_periods=1).sum()

# 6. Racha de viento reciente (Pico de racha en los últimos 3 días)
wind_speed_ds['fg_3d_max'] = wind_speed_ds['fg'].rolling(time=3, min_periods=1).max()

## Dataset Building

In [9]:
NEGATIVES_RATIO = 3

target_dates_neg = df['Detectado'].repeat(NEGATIVES_RATIO).reset_index(drop=True)

np.random.seed(42)
valid_chunks = []
total_needed = len(target_dates_neg)
current_count = 0

while current_count < total_needed:
    left = total_needed - current_count

    rand_lons = np.random.uniform(min_lon, max_lon, size=left)
    rand_lats = np.random.uniform(min_lat, max_lat, size=left)
    
    batch_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(rand_lons, rand_lats),
        crs="EPSG:4326"
    )
    
    in_forest = gpd.sjoin(batch_gdf, mfe, how="inner", predicate="within")
    
    if not in_forest.empty:
        valid_chunks.append(in_forest)
        current_count += len(in_forest)

valid_negatives = pd.concat(valid_chunks).iloc[:total_needed].copy()

valid_negatives['Detectado'] = target_dates_neg.values
valid_negatives['Longitude'] = valid_negatives.geometry.x
valid_negatives['Latitude'] = valid_negatives.geometry.y
valid_negatives['Label'] = 0

negatives_df = pd.DataFrame(valid_negatives[['Detectado', 'Longitude', 'Latitude', 'Label']])

df["Label"] = 1
df = pd.concat([df, negatives_df], ignore_index=True)

df = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]), crs="EPSG:4326").sort_values(by='Detectado').reset_index(drop=True)

Positive Points Dataset

In [10]:
full_df = gpd.sjoin(df, mfe, how="left", predicate="within").drop(columns=["index_right", "geometry"])

lon = xr.DataArray(full_df["Longitude"].values, dims="points")
lat = xr.DataArray(full_df["Latitude"].values, dims="points")
time = xr.DataArray(full_df["Detectado"].values, dims="points")

climate_datasets = [rain_ds, rel_humidity_ds, max_temperature_ds, avg_temperature_ds, radiation_ds, wind_speed_ds]

for ds in climate_datasets:
    sampled = ds.sel(longitude=lon, latitude=lat, method="nearest").sel(time=time, method="pad")
    for var in ds.data_vars:
        full_df[var] = sampled[var].values

full_df["elevation"] = elevation_ds["elevation"].sel(longitude=lon, latitude=lat, method="nearest").values

In [12]:
print(full_df.columns)
full_df.to_csv("./dataset.csv")

Index(['Detectado', 'Longitude', 'Latitude', 'Label', 'Superficie', 'FCCARB',
       'FCCMAT', 'FCCHER', 'FCCTOT', 'HMMAT', 'O1', 'O2', 'O3', 'TipesTR25',
       'n_MODCOM', 'n_REGBIO', 'Riesgo_Especie1', 'Riesgo_Especie2',
       'Riesgo_Especie3', 'Estado1_num', 'Estado2_num', 'Estado3_num',
       'TIPOBOSQUE_num', 'Distribuci_num', 'rr', 'rr_7d_sum', 'rr_30d_sum',
       'rr_60d_sum', 'rr_90d_sum', 'hu', 'hu_7d_mean', 'hu_14d_mean',
       'hu_30d_mean', 'tx', 'tx_3d_max', 'tx_7d_mean', 'tx_14d_mean',
       'tx_30d_mean', 'tg', 'tg_7d_mean', 'tg_30d_mean', 'qq', 'qq_7d_sum',
       'qq_30d_sum', 'fg', 'fg_3d_max', 'elevation'],
      dtype='str')


In [13]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)